In [ ]:
import os
import mesa
import google.generativeai as genai
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [ ]:
import json
import random
import os
import mesa
import google.generativeai as genai

api_key = os.environ.get("GOOGLE_API_KEY") 
genai.configure(api_key=api_key)

generation_config = genai.GenerationConfig(
    response_mime_type="application/json",
)
generation_model = genai.GenerativeModel('gemini-2.5-flash', generation_config=generation_config)

class LLMAgent(mesa.Agent):
    def __init__(self, model):
        super().__init__(model)
        self.wealth = 1
        self.memory = [] 

    def step(self):
        other_agents = [a.unique_id for a in self.model.agents if a.unique_id != self.unique_id]
        
        recent_memory = self.memory[-3:] if self.memory else "No past memories."
    
        prompt = f"""
        You are Agent {self.unique_id} in an economic simulation. 
        You have {self.wealth} units of wealth.
        Your recent memory: {recent_memory}
        
        You must give 1 unit of wealth to another agent. The available agents are: {other_agents}.
        
        Respond STRICTLY with a JSON object containing:
        1. "target_id": The ID of the agent you choose to give money to (must be an integer from {other_agents}).
        2. "dialogue": A dramatic, theatrical villain quote written STRICTLY IN ENGLISH, explaining why you chose them.
        """
        
        try:
           
            response = generation_model.generate_content(prompt)
            decision = json.loads(response.text)
            
            target_id = decision.get("target_id")
            dialogue = decision.get("dialogue")
            
           
            if self.wealth > 0 and target_id in other_agents:
                target_agent = self.model.agents.select(lambda a: a.unique_id == target_id)[0]
                target_agent.wealth += 1
                self.wealth -= 1
                
               
                action_log = f"I gave 1 wealth to Agent {target_id}."
                self.memory.append(action_log)
                
                print(f"Agent {self.unique_id} (Wealth: {self.wealth}) -> Gave to Agent {target_id}: '{dialogue}'")
            else:
                print(f"Agent {self.unique_id} is broke and plotting revenge.")
                self.memory.append("I was broke and could do nothing.")
                
        except Exception as e:
            print(f"Agent {self.unique_id} API Error/Parsing Error: {e}")

class LLMModel(mesa.Model):
    def __init__(self, n):
        super().__init__()
        self.num_agents = n
        LLMAgent.create_agents(model=self, n=n)

    def step(self):
        self.agents.shuffle_do("step")

In [16]:
test_model = LLMModel(3)
test_model.step()

Agent 2 (Wealth: 0) -> Gave to Agent 1: 'Ah, Agent 1. A pawn on my chessboard, perhaps? Or merely a convenient stepping stone in my grand design. Take this, little one; every coin has its purpose, and yours, for now, is merely to fuel my ascent!'
Agent 3 (Wealth: 0) -> Gave to Agent 2: 'Agent 2, you have been singled out for a peculiar honor, a mere morsel of my 'generosity' that shall bind you ever closer to the shadows of my influence!'
Agent 1 (Wealth: 1) -> Gave to Agent 2: 'Ah, Agent 2, your fate is sealed by my whim! This pittance is but a taste of the grand design I have for you. Do not mistake this gesture for generosity; it is merely the first thread in the web I weave.'
